In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [3]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [4]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'B-1Density',
        'B-2Density',
        'B0Share',
        'B1Share',
        'B-1Share',
        'B-2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare'
    ]


In [5]:
query = f"""DROP VIEW IF EXISTS Weekly_KPI;"""
cursor.execute(query)
conn.commit()


In [6]:
query = """
CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Weekly'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name;"""
cursor.execute(query)
conn.commit()

In [7]:
query = "SELECT * FROM Weekly_KPI WHERE Year = 2026;"
df = pd.read_sql_query(query, conn)
conn.close()

In [8]:
df

,Year,PeriodValue,CustomerName,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,ServicePipeKm,ServicePipeCoveredKm,ReportCount,...,B1Density,B-1Density,B-2Density,B0Share,B1Share,B-1Share,B-2Share,NGShare,PGShare,Not_NGShare
0,2026,1,Cadent,50.53,47.68,49.69,47.03,0.84,0.65,2.0,...,0.00,1.42,0.06,0.23,0.00,0.74,0.03,0.73,0.19,0.07
1,2026,2,Cadent,229.57,216.26,226.35,213.85,3.22,2.41,9.0,...,0.01,0.70,0.06,0.29,0.01,0.65,0.05,0.68,0.22,0.10
2,2026,3,Cadent,312.37,282.81,302.16,276.15,10.21,6.67,11.0,...,0.03,0.60,0.04,0.38,0.03,0.55,0.04,0.73,0.19,0.08
3,2026,4,Cadent,339.23,321.31,332.75,315.89,6.48,5.42,12.0,...,0.01,0.76,0.07,0.24,0.01,0.69,0.06,0.72,0.21,0.07
4,2026,5,Cadent,716.21,671.67,699.31,657.47,16.90,14.20,25.0,...,0.01,0.56,0.07,0.22,0.01,0.69,0.08,0.63,0.27,0.10
5,2026,6,Cadent,1530.09,1412.45,1491.98,1383.53,38.11,28.92,54.0,...,0.01,0.73,0.09,0.26,0.01,0.65,0.08,0.69,0.22,0.10
6,2026,7,Cadent,1442.95,1327.13,1408.33,1303.17,34.62,23.95,53.0,...,0.01,0.73,0.10,0.21,0.01,0.69,0.10,0.70,0.20,0.10
7,2026,8,Cadent,1647.92,1560.83,1618.80,1539.95,29.12,20.88,54.0,...,0.00,0.59,0.07,0.26,0.01,0.66,0.08,0.73,0.21,0.06
8,2026,9,Cadent,1466.42,1393.40,1435.40,1369.36,31.02,24.03,52.0,...,0.01,0.62,0.09,0.24,0.01,0.66,0.09,0.75,0.17,0.08
9,2026,10,Cadent,1210.19,1134.14,1191.31,1119.10,18.87,15.04,43.0,...,0.02,0.79,0.15,0.17,0.02,0.68,0.13,0.72,0.15,0.13


In [9]:
df.to_excel("pivot_view.xlsx")

In [10]:
Query(query = f"SELECT * FROM KPI_Data WHERE Year = 2026 AND PeriodValue = '23' AND CustomerId = (SELECT CustomerId FROM KPI_Customer WHERE Name = 'Cadent')").execute(KPIHub_Conn)

,Id,KPIId,CustomerId,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_Y2026_W23,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,0.94,None,2026-06-22 09:57:37.990053
1,ReportAssetLengthKm_Cadent_Y2026_W23,ReportAssetLengthKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,3455.5,None,2026-06-22 09:57:37.990053
2,AssetCoveredLengthKm_Cadent_Y2026_W23,AssetCoveredLengthKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,3231.83,None,2026-06-22 09:57:37.990053
3,DistributionPipeKm_Cadent_Y2026_W23,DistributionPipeKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,3376.26,None,2026-06-22 09:57:37.990053
4,DistributionPipeCoveredKm_Cadent_Y2026_W23,DistributionPipeCoveredKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,3171.7,None,2026-06-22 09:57:37.990053
5,ServicePipeKm_Cadent_Y2026_W23,ServicePipeKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,79.24,None,2026-06-22 09:57:37.990053
6,ServicePipeCoveredKm_Cadent_Y2026_W23,ServicePipeCoveredKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,60.12,None,2026-06-22 09:57:37.990053
7,ReportCount_Cadent_Y2026_W23,ReportCount,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,104.0,None,2026-06-22 09:57:37.990053
8,LisaCount_Cadent_Y2026_W23,LisaCount,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,3498.0,None,2026-06-22 09:58:07.100715
9,EmissionRate_Cadent_Y2026_W23,EmissionRate,BD4D080B-1D12-D329-ABD0-39FEB9804E98,2026,Weekly,23,5423.78,None,2026-06-22 09:58:07.100715
